# **Install necessary libraries**

In [ ]:
# Unzip the library
!unzip unsloth.zip

# Go inside the directory
%cd /content/unsloth

# Install unsloth library with the configuration
!pip install -e ".[cu128-torch280]"

# Install flash-attention-2
!pip install flash-attn --no-build-isolation --no-cache-dir \
    --find-links https://github.com/Dao-AILab/flash-attention/releases/tag/v2.8.3

# **Load Libraries**

In [ ]:
import torch
from unsloth import FastLanguageModel

from datasets import (load_dataset, concatenate_datasets,
                      Dataset, DatasetDict)

from tqdm.notebook import tqdm
from typing import Dict, List, Optional, Any

# **Initialize Model**

In [ ]:
# Configure model settings
model_name = "taharmasmaliyev07/Qwen-3-4B-b16-tuned-full"
MAX_SEQ_LENGTH = 20000

In [ ]:
# Load the model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = False,
    attn_implementation="flash_attention_2",
    dtype=torch.bfloat16,
)

# Switch model to inference mode
FastLanguageModel.for_inference(model)

# **Inference functions**

In [ ]:
def run_inference(
    instruction: str,
    max_new_tokens: int = 2048,
    temperature: float = 0.7,
    top_p: float = 0.9,
    do_sample: bool = True,
) -> str:
    """
    Run inference for a single instruction.

    Args:
        instruction:    The user question
        max_new_tokens: Maximum number of tokens to generate.
        temperature:    Sampling temperature (lower = more deterministic).
        top_p:          Nucleus sampling probability.
        do_sample:      Whether to sample; set False for greedy decoding.

    Returns:
        The model's response string (with special tokens preserved).
    """
    messages = [{'role': 'user', 'content': instruction}]

    # Build the prompt string via chat template
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    # Tokenize and move to device
    inputs = tokenizer(
        prompt_text,
        return_tensors='pt',
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=do_sample,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens (skip the prompt)
    generated_ids = output_ids[0][inputs['input_ids'].shape[-1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=False)
    return response


def run_batch_inference(
    instructions: List[str],
    max_new_tokens: int = 2048,
    temperature: float = 0.7,
    top_p: float = 0.9,
    do_sample: bool = True,
) -> List[str]:
    """
    Run inference for a batch of instructions.

    Args:
        instructions:   List of user question / task prompts.
        max_new_tokens: Maximum number of tokens to generate per sample.
        temperature:    Sampling temperature.
        top_p:          Nucleus sampling probability.
        do_sample:      Whether to sample; set False for greedy decoding.

    Returns:
        List of response strings (one per instruction).
    """
    prompt_texts = [
        tokenizer.apply_chat_template(
            [{'role': 'user', 'content': instr}],
            tokenize=False,
            add_generation_prompt=True,
        )
        for instr in instructions
    ]

    inputs = tokenizer(
        prompt_texts,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=do_sample,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    prompt_lengths = inputs['input_ids'].shape[-1]
    responses = [
        tokenizer.decode(out[prompt_lengths:], skip_special_tokens=False)
        for out in output_ids
    ]
    return responses

# **Run Inference**

In [ ]:
# ── Single instruction example ────────────────────────────────────────────────
instruction = "Solve the following equation and explain each step: 3x + 7 = 22"

response = run_inference(instruction, max_new_tokens=1024, temperature=0.7)
print("=" * 60)
print("INSTRUCTION:")
print(instruction)
print("=" * 60)
print("RESPONSE:")
print(response)

In [ ]:
# ── Batch inference example ───────────────────────────────────────────────────
instructions = [
    "What is the time complexity of binary search and why?",
    "Write a Python function that checks whether a number is prime.",
]

responses = run_batch_inference(instructions, max_new_tokens=1024, temperature=0.7)
for instr, resp in zip(instructions, responses):
    print("=" * 60)
    print("INSTRUCTION:", instr)
    print("-" * 60)
    print("RESPONSE:")
    print(resp)
    print()